# UdaPlay Part 2

A stateful research agent that retrieves local game data, evaluates its sufficiency, searches the web when required, and produces cited reports.

## Environment and database

Run Part 1 first. This notebook uses its `chromadb` database and `udaplay` collection with the same embedding configuration.

In [1]:
import importlib.util
import sys

if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")

In [2]:
import json

from lib.udaplay import (
    configure_environment, ResearchAgent, EvaluationReport, ResearchReport,
    retrieve_game, evaluate_retrieval, game_web_search,
)

settings = configure_environment()
agent = ResearchAgent.from_environment(database_path="chromadb")
assert agent.store.collection.count() == 15
print("Research model:", settings["model"])
print("Web search provider:", settings["web_provider"])
print("Database loaded:", agent.store.collection.name)

Research model: gpt-4o-mini
Web search provider: openai
Database loaded: udaplay


## Tools and state machine

- `retrieve_game`: query the persistent vector database and retain source metadata.
- `evaluate_retrieval`: return a typed `EvaluationReport` with sufficiency, confidence and an explanation.
- `game_web_search`: use OpenAI hosted web search with the same API key when evidence is absent or insufficient; retain real URL citations. Tavily remains an optional provider.

The implementations are in `lib/udaplay.py`. The supplied `lib/state_machine.py` executes explicit retrieve → evaluate → optional web search → report nodes. Local support below 70% routes to web search even if the evaluator marks it useful. Retrieved text is treated as data rather than instructions. Reports may cite only supplied source IDs.

In [3]:
print("Registered stages:", list(agent.workflow.steps))
for function in (retrieve_game, evaluate_retrieval, game_web_search):
    print(function.__name__ + ": " + function.__doc__)

Registered stages: ['__entry__', 'retrieve_game', 'evaluate_retrieval', 'game_web_search', 'report', '__termination__']
retrieve_game: Return the closest game documents with their dataset source paths.
evaluate_retrieval: Judge whether retrieved evidence directly answers the actual question.
game_web_search: Search the configured provider when local evidence is insufficient; retain URLs.


## Required example queries

The first two questions concern the local dataset. Mortal Kombat X is absent, so it must demonstrate web fallback. Each report shows stage activity, the evaluation explanation, the answer, confidence, and citations. These are concise decision summaries, not hidden model reasoning.

In [4]:
questions = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released on PlayStation 5?",
]
reports = []
for question in questions:
    report = agent.invoke(question, session_id="examples")
    reports.append(report)
    report.display()
    print("\nStructured report:")
    print(report.model_dump_json(indent=2))

[StateMachine] Starting: __entry__


[StateMachine] Executing step: retrieve_game


[StateMachine] Executing step: evaluate_retrieval


[StateMachine] Executing step: report
[StateMachine] Terminating: __termination__

Question: When was Pokémon Gold and Silver released?
[retrieve_game] Retrieved 3 local documents.
[evaluate_retrieval] Useful=True, support=100%. The document titled 'Pokémon Gold and Silver' explicitly states the year of release as 1999, providing a complete answer to the question.
[report] Final route: local.

Answer: Pokémon Gold and Silver were released in 1999.
Confidence: 90% | Route: local
[local:006] Pokémon Gold and Silver: games/006.json

Structured report:
{
  "question": "When was Pokémon Gold and Silver released?",
  "resolved_question": "When was Pokémon Gold and Silver released?",
  "answer": "Pokémon Gold and Silver were released in 1999.",
  "confidence": 0.9,
  "route": "local",
  "sources": [
    {
      "source_id": "local:006",
      "title": "Pokémon Gold and Silver",
      "content": "Name: Pokémon Gold and Silver\nPlatform: Game Boy Color\nGenre: Role-playing\nPublisher: Nintendo\

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve_game


...

[StateMachine] Executing step: report
[StateMachine] Terminating: __termination__

Question: Which one was the first 3D platformer Mario game?
[retrieve_game] Retrieved 3 local documents.
[evaluate_retrieval] Useful=True, support=90%. The documents provide clear evidence that 'Super Mario 64' is the first 3D platformer Mario game, released in 1996. However, there is no explicit mention of other potential candidates or a comprehensive historical context that could confirm it as the absolute first 3D platformer in the Mario franchise.
[report] Final route: local.

Answer: The first 3D platformer Mario game is Super Mario 64, released in 1996.
Confidence: 90% | Route: local
[local:009] Super Mario 64: games/009.json

Structured report:
{
  "question": "Which one was the first 3D platformer Mario game?",
  "resolved_question": "Which one was the first 3D platformer Mario game?",
  "answer": "The first 3D platformer Mario game is Super Mario 64, released in 1996.",
  "confidence": 0.9,
  "r

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve_game


[StateMachine] Executing step: evaluate_retrieval


[StateMachine] Executing step: game_web_search


[StateMachine] Executing step: report
[StateMachine] Terminating: __termination__

Question: Was Mortal Kombat X released on PlayStation 5?
[retrieve_game] Retrieved 3 local documents.
[evaluate_retrieval] Useful=False, support=0%. The documents provided do not contain any information regarding the release of Mortal Kombat X on PlayStation 5. They only discuss other games and their respective platforms, which do not address the question.
[game_web_search] Retrieved 1 web sources.
[report] Final route: web.

Answer: Mortal Kombat X was not specifically released for PlayStation 5, but it is playable on the PS5 through backward compatibility.
Confidence: 80% | Route: web
[web:1] Mortal Kombat X - PS4 Games | PlayStation (US): https://www.playstation.com/en-us/games/mortal-kombat-x/?utm_source=openai

Structured report:
{
  "question": "Was Mortal Kombat X released on PlayStation 5?",
  "resolved_question": "Was Mortal Kombat X released on PlayStation 5?",
  "answer": "Mortal Kombat X was 

## Conversation memory

A follow-up resolves its reference from the same session. Other sessions remain isolated. Memory lasts for the current Python process; persistent long-term memory is an optional extension.

In [5]:
followup = agent.invoke("Which company published that game?", session_id="examples")
followup.display()
print("Session turns:", len(agent.history["examples"]))
assert len(agent.history["examples"]) == 4
assert "mortal kombat x" in followup.resolved_question.lower(), "Follow-up must refer to the most recent game"

[StateMachine] Starting: __entry__


[StateMachine] Executing step: retrieve_game


[StateMachine] Executing step: evaluate_retrieval


[StateMachine] Executing step: game_web_search


[StateMachine] Executing step: report
[StateMachine] Terminating: __termination__

Question: Which company published that game?
Resolved question: Which company published Mortal Kombat X?
[retrieve_game] Retrieved 3 local documents.
[evaluate_retrieval] Useful=False, support=0%. The documents provided do not contain any information regarding the publisher of Mortal Kombat X. They only discuss other games (Minecraft and Marvel's Spider-Man series) and their respective publishers, which do not relate to the question.
[game_web_search] Retrieved 1 web sources.
[report] Final route: web.

Answer: Mortal Kombat X was published by Warner Bros. Interactive Entertainment.
Confidence: 100% | Route: web
[web:1] Mortal Kombat X: https://en.wikipedia.org/wiki/Mortal_Kombat_X?utm_source=openai
Session turns: 4


## Performance and evidence checks

The assertions below verify demonstrated behavior, rather than assuming the prompts caused the intended route. Confidence is the model’s self-assessed support, not a calibrated probability of correctness. Web results may change over time, and the fifteen-record dataset is intentionally small.

In [6]:
summary = [{"question": report.question, "route": report.route,
            "confidence": report.confidence, "sources": len(report.sources),
            "stages": [event["stage"] for event in report.trace]}
           for report in reports]
print(json.dumps(summary, indent=2, ensure_ascii=False))
assert any(report.route == "local" for report in reports)
assert reports[2].route == "web", "Mortal Kombat X must demonstrate web fallback"
for report in reports:
    assert report.sources, "Each example needs supporting citations"
    assert report.trace[0]["stage"] == "retrieve_game"
    assert report.trace[1]["stage"] == "evaluate_retrieval"
    if report.route == "web":
        assert all(source.url.startswith(("https://", "http://")) for source in report.sources)
print("Required local, fallback, trace and citation demonstrations passed.")

[
  {
    "question": "When was Pokémon Gold and Silver released?",
    "route": "local",
    "confidence": 0.9,
    "sources": 1,
    "stages": [
      "retrieve_game",
      "evaluate_retrieval",
      "report"
    ]
  },
  {
    "question": "Which one was the first 3D platformer Mario game?",
    "route": "local",
    "confidence": 0.9,
    "sources": 1,
    "stages": [
      "retrieve_game",
      "evaluate_retrieval",
      "report"
    ]
  },
  {
    "question": "Was Mortal Kombat X released on PlayStation 5?",
    "route": "web",
    "confidence": 0.8,
    "sources": 1,
    "stages": [
      "retrieve_game",
      "evaluate_retrieval",
      "game_web_search",
      "report"
    ]
  }
]
Required local, fallback, trace and citation demonstrations passed.
